# CheckM2 Evaluation on Fecal Metagenome Assembly

Evaluate genome bin quality using CheckM2 (completeness & contamination)
on real assembled contigs from a fecal metagenome.

**Pipeline**: Load FASTA contigs → Embed with trained models → Cluster → Export bins as FASTA → Run CheckM2

**Models**: NonLinear, Hinge, Bern-NT, UncertainGen, Bern-NT+UG, PCL, LLA
**Clustering**: Greedy KMedoid, KMeans, DPGMM
**Data**: `data/Fecal/eukfilt_assembly.fasta` (17,444 assembled contigs)

In [1]:
import sys, os, csv
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.cluster import KMeans
from sklearn.mixture import BayesianGaussianMixture
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from embedders.nonlinear import NonLinearEmbedder, set_seed
from embedders.pcl import PCLEmbedder
from embedders.uncertaingen import UncertainGenEmbedder
from embedders.laplace_embedder import LaplaceLastLayerEmbedder
from embedders.base import EmbeddingResult
from clustering.greedy_kmedoid import GreedyKMedoidClusterer
from evaluation.eval_utils import compute_class_center_medium_similarity
from evaluation.checkm2 import evaluate_checkm2, export_bins_to_fasta, CheckM2Summary

# ── Config ──
SEED         = 26042024
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_DIR    = '../../models/notebook/unlabeled_exp'
FASTA_PATH   = '../../data/Fecal/eukfilt_assembly.fasta'
METRIC       = 'l2'
MIN_BIN_SIZE = 5
K, DIM       = 4, 256

# Embedding uses truncated sequences; CheckM2 uses full contigs
MAX_SEQ_LEN  = 20000
MIN_SEQ_LEN  = 2500

CHECKM2_CMD     = 'checkm2'
CHECKM2_THREADS = 8
CHECKM2_DB      = os.path.abspath('../../checkm2/database/CheckM2_database/uniref100.KO.1.dmnd')
os.environ['CHECKM2DB'] = CHECKM2_DB

# Reference dataset for threshold calibration (Greedy KMedoid)
REF_TSV = '../../data/dnabert/eval/reference/binning_5.tsv'

MODEL_NAMES  = ['nl', 'hinge', 'bern_nt', 'ug', 'ug_bnt', 'pcl', 'lla']
MODEL_LABELS = {
    'nl': 'NonLinear', 'hinge': 'Hinge', 'bern_nt': 'Bern-NT',
    'ug': 'UncertainGen', 'ug_bnt': 'Bern-NT+UG',
    'pcl': 'PCL', 'lla': 'LLA',
}

# Models sharing the same mean network produce identical point_estimate,
# so clustering/thresholds/CheckM2 are identical. We compute once per group.
MEAN_GROUPS = {
    'nl': 'nl_group',       'ug': 'nl_group',      'lla': 'nl_group',
    'hinge': 'hinge_group',
    'bern_nt': 'bnt_group', 'ug_bnt': 'bnt_group',
    'pcl': 'pcl_group',
}
GROUP_REP = {
    'nl_group': 'nl', 'hinge_group': 'hinge',
    'bnt_group': 'bern_nt', 'pcl_group': 'pcl',
}
UNIQUE_MODELS = list(GROUP_REP.values())  # ['nl', 'hinge', 'bern_nt', 'pcl']

# KMeans k values (from clustering_evaluation.ipynb)
BEST_K_KMEANS = {
    'nl': 500, 'hinge': 450, 'bern_nt': 500,
    'ug': 500, 'ug_bnt': 500,
    'pcl': 500, 'lla': 500,
}
BEST_PCA_DIM         = 16
DPGMM_MAX_COMPONENTS = 500

print(f'Device: {DEVICE}')
print(f'CheckM2 DB: {CHECKM2_DB}')
print(f'Unique mean groups: {UNIQUE_MODELS} (skipping redundant: ug, ug_bnt, lla)')

Device: cuda
CheckM2 DB: r:\revisitingkmers\checkm2\database\CheckM2_database\uniref100.KO.1.dmnd
Unique mean groups: ['nl', 'hinge', 'bern_nt', 'pcl'] (skipping redundant: ug, ug_bnt, lla)


## 1. Load Fecal Metagenome Contigs

Parse `eukfilt_assembly.fasta`. Keep contigs ≥ 2500 bp.
- **Truncated sequences** (≤ 20,000 bp) are used for embedding (model input).
- **Full sequences** are used for CheckM2 export (genome completeness needs all bases).

In [2]:
def read_fasta(path, max_seq_len=None, min_seq_len=0):
    """Parse FASTA, filter by length. Returns (names, full_seqs, truncated_seqs)."""
    names, full_seqs, trunc_seqs = [], [], []
    name, seq_parts = None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if name:
                    full = ''.join(seq_parts)
                    if len(full) >= min_seq_len:
                        names.append(name)
                        full_seqs.append(full)
                        trunc_seqs.append(full[:max_seq_len] if max_seq_len else full)
                name = line[1:].split()[0]
                seq_parts = []
            else:
                seq_parts.append(line)
    if name:
        full = ''.join(seq_parts)
        if len(full) >= min_seq_len:
            names.append(name)
            full_seqs.append(full)
            trunc_seqs.append(full[:max_seq_len] if max_seq_len else full)
    return names, full_seqs, trunc_seqs

contig_names, full_sequences, embed_sequences = read_fasta(
    FASTA_PATH, max_seq_len=MAX_SEQ_LEN, min_seq_len=MIN_SEQ_LEN)

lengths = np.array([len(s) for s in full_sequences])
print(f'Loaded {len(contig_names)} contigs (min_len={MIN_SEQ_LEN} bp)')
print(f'  Length stats: min={lengths.min()}, median={int(np.median(lengths))}, '
      f'max={lengths.max()}, total={lengths.sum()/1e6:.0f} Mbp')
print(f'  Truncated to ≤{MAX_SEQ_LEN} bp for embedding')

Loaded 17444 contigs (min_len=2500 bp)
  Length stats: min=3002, median=30207, max=6424695, total=1731 Mbp
  Truncated to ≤20000 bp for embedding


## 2. Load Models & Embed

In [3]:
# ── Load models ──
model_nl = NonLinearEmbedder.load(os.path.join(MODEL_DIR, 'nonlinear_unlabeled.model'), device=DEVICE)
model_hinge = NonLinearEmbedder.load(os.path.join(MODEL_DIR, 'hinge_unlabeled.model'), device=DEVICE)
model_bern_nt = NonLinearEmbedder.load(os.path.join(MODEL_DIR, 'bern_nt_unlabeled.model'), device=DEVICE)
model_ug = UncertainGenEmbedder.load(os.path.join(MODEL_DIR, 'uncertaingen_unlabeled.model'), device=DEVICE)
model_ug_bnt = UncertainGenEmbedder.load(os.path.join(MODEL_DIR, 'ug_bern_nt_unlabeled.model'), device=DEVICE)
model_pcl = PCLEmbedder.load(os.path.join(MODEL_DIR, 'pcl_unlabeled2.model'), device=DEVICE)

# LLA wraps the NL model
model_lla = LaplaceLastLayerEmbedder(model_nl)
lla_state = torch.load(os.path.join(MODEL_DIR, 'lla_state.pt'), map_location=DEVICE, weights_only=True)
model_lla.Q_A = lla_state['Q_A']
model_lla.Q_B = lla_state['Q_B']
model_lla.S_A = lla_state['S_A']
model_lla.S_B = lla_state['S_B']
model_lla.prior_precision = lla_state['prior_precision']
model_lla.n_data = lla_state['n_data']
model_lla._fitted = True

models = {
    'nl': model_nl, 'hinge': model_hinge, 'bern_nt': model_bern_nt,
    'ug': model_ug, 'ug_bnt': model_ug_bnt,
    'pcl': model_pcl, 'lla': model_lla,
}
print('All models loaded.')

All models loaded.


In [4]:
# ── Embed contigs (using truncated sequences) ──
print('Embedding contigs...')
contig_embs = {}
for m in MODEL_NAMES:
    print(f'  {MODEL_LABELS[m]}...', end=' ', flush=True)
    contig_embs[m] = models[m].embed(embed_sequences)
    e = contig_embs[m]
    extra = ''
    if e.variance is not None:
        extra += f'  var_mean={e.variance.mean():.5f}'
    if e.kappa is not None:
        extra += f'  kappa=[{e.kappa.min():.1f}, {e.kappa.max():.1f}]'
    print(f'{e.mean.shape}{extra}')

Embedding contigs...
  NonLinear... (17444, 256)
  Hinge... (17444, 256)
  Bern-NT... (17444, 256)
  UncertainGen... (17444, 256)  var_mean=0.00038
  Bern-NT+UG... (17444, 256)  var_mean=0.04565
  PCL... (17444, 256)  kappa=[337.2, 868.4]
  LLA... (17444, 256)  var_mean=0.00222


## 3. Compute Thresholds (for Greedy KMedoid)

Use reference dataset (binning_5.tsv) validation embeddings to calibrate similarity thresholds.
The fecal data has no ground-truth labels, so we transfer thresholds from the labeled reference set.

In [5]:
# Load reference validation data for threshold calibration
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

with open(REF_TSV) as f:
    reader = csv.reader(f, delimiter='\t')
    _header = next(reader)
    all_rows = list(reader)

ref_seqs       = [r[0] for r in all_rows]
ref_labels_str = [r[1] for r in all_rows]

val_seqs, _, val_labels_str, _ = train_test_split(
    ref_seqs, ref_labels_str, test_size=0.5, random_state=SEED, stratify=ref_labels_str)

unique_labels = sorted(set(ref_labels_str))
lab2id = {l: i for i, l in enumerate(unique_labels)}
val_labels = np.array([lab2id[l] for l in val_labels_str])

print(f'Reference val set: {len(val_seqs)} sequences, {len(unique_labels)} species')

# Embed reference validation set only for unique mean groups (4 instead of 7)
print('Embedding reference validation set (unique mean groups only)...')
val_embs = {}
for m in UNIQUE_MODELS:
    print(f'  {MODEL_LABELS[m]}...', end=' ', flush=True)
    val_embs[m] = models[m].embed(val_seqs)
    print(f'{val_embs[m].mean.shape}')

# Compute thresholds only for unique mean groups
thresholds = {}
scales = {}

for m in UNIQUE_MODELS:
    e = val_embs[m]
    kwargs = dict(metric=METRIC)
    if m == 'pcl':
        kwargs['kappas'] = e.kappa
        kwargs['k_form'] = 'cosine_direct'
        kwargs['alpha'] = 1.0

    pv, sc = compute_class_center_medium_similarity(
        e.point_estimate, val_labels, **kwargs)

    idx = -1 if m == 'pcl' else -3
    thresholds[m] = pv[idx]
    scales[m] = sc
    print(f'{MODEL_LABELS[m]:>12s}: threshold={thresholds[m]:.4f}  scale={scales[m]:.4f}')

# Copy thresholds/scales for models sharing the same mean
for m in MODEL_NAMES:
    if m not in thresholds:
        rep = GROUP_REP[MEAN_GROUPS[m]]
        thresholds[m] = thresholds[rep]
        scales[m] = scales[rep]
        print(f'{MODEL_LABELS[m]:>12s}: copied from {MODEL_LABELS[rep]} '
              f'(threshold={thresholds[m]:.4f}  scale={scales[m]:.4f})')

# Free reference data
del val_embs, val_seqs, ref_seqs, all_rows

Reference val set: 18639 sequences, 323 species
Embedding reference validation set (unique mean groups only)...
  NonLinear... (18639, 256)
  Hinge... (18639, 256)
  Bern-NT... (18639, 256)
  PCL... (18639, 256)
Auto-calibrated scale: 2.325874 (median raw distance: 0.2980)
Percentile values: [0.13469695948590307, 0.2478129589322687, 0.3440444825228076, 0.42430783104547165, 0.49999996948309794, 0.5718823524604251, 0.6409133654115231, 0.7055413762306159, 0.7809424521158268]
   NonLinear: threshold=0.6409  scale=2.3259
Auto-calibrated scale: 7.413558 (median raw distance: 0.0935)
Percentile values: [0.15069984065974498, 0.2547610869942971, 0.34646003973295564, 0.42783402049195346, 0.49999999760358044, 0.5692493707381968, 0.6369857024645751, 0.7022794414774678, 0.7757844747543642]
       Hinge: threshold=0.6370  scale=7.4136
Auto-calibrated scale: 4.238681 (median raw distance: 0.1635)
Percentile values: [0.17796589256655931, 0.27335453179281033, 0.3534285942876704, 0.4296497934761457, 0.4

## 4. Clustering

Run Greedy KMedoid, KMeans, and DPGMM on fecal contig embeddings.

In [6]:
N = len(contig_names)
cluster_results = {}  # (model, method) -> pred labels

# ── Greedy KMedoid (run only for unique mean groups) ──
print('=== Greedy KMedoid ===')
for m in UNIQUE_MODELS:
    e = contig_embs[m]
    sub_emb = EmbeddingResult(mean=e.mean, variance=None,
                              kappa=e.kappa if e.kappa is not None else None)
    kwargs = dict(metric=METRIC, min_bin_size=MIN_BIN_SIZE, scale=scales[m])
    if m == 'pcl':
        kwargs['k_form'] = 'cosine_direct'
        kwargs['alpha'] = 1.0

    gkmed = GreedyKMedoidClusterer(**kwargs)
    pred = gkmed.fit_predict(sub_emb, min_similarity=thresholds[m])
    cluster_results[(m, 'greedy_kmedoid')] = pred

    n_clusters = len(set(pred[pred != -1].tolist())) if (pred != -1).any() else 0
    assigned = int((pred != -1).sum())
    print(f'  {MODEL_LABELS[m]:>12s}: k={n_clusters:4d}  assigned={assigned:5d}/{N}')

# Copy Greedy KMedoid results for shared-mean models
for m in MODEL_NAMES:
    if m not in UNIQUE_MODELS:
        rep = GROUP_REP[MEAN_GROUPS[m]]
        cluster_results[(m, 'greedy_kmedoid')] = cluster_results[(rep, 'greedy_kmedoid')]
        print(f'  {MODEL_LABELS[m]:>12s}: copied from {MODEL_LABELS[rep]}')

# ── KMeans (run only for unique mean groups) ──
print('\n=== KMeans ===')
for m in UNIQUE_MODELS:
    X = contig_embs[m].point_estimate
    best_k = BEST_K_KMEANS[m]
    km = KMeans(n_clusters=best_k, random_state=SEED, n_init=5, max_iter=300)
    pred = km.fit_predict(X)

    bin_counts = Counter(pred)
    small = {c for c, n in bin_counts.items() if n < MIN_BIN_SIZE}
    if small:
        pred = np.where(np.isin(pred, list(small)), -1, pred)

    cluster_results[(m, 'kmeans')] = pred
    n_clusters = len(set(pred[pred != -1].tolist())) if (pred != -1).any() else 0
    assigned = int((pred != -1).sum())
    print(f'  {MODEL_LABELS[m]:>12s}: k={n_clusters:4d}  assigned={assigned:5d}/{N}')

# Copy KMeans results for shared-mean models
for m in MODEL_NAMES:
    if m not in UNIQUE_MODELS:
        rep = GROUP_REP[MEAN_GROUPS[m]]
        cluster_results[(m, 'kmeans')] = cluster_results[(rep, 'kmeans')]
        print(f'  {MODEL_LABELS[m]:>12s}: copied from {MODEL_LABELS[rep]}')

# ── DPGMM (run only for unique mean groups) ──
print('\n=== DPGMM ===')
for m in UNIQUE_MODELS:
    X = contig_embs[m].point_estimate
    pca_dim = min(BEST_PCA_DIM, X.shape[0] - 1, X.shape[1])
    X_pca = PCA(n_components=pca_dim, random_state=SEED).fit_transform(X)

    n_comp = min(DPGMM_MAX_COMPONENTS, N - 1)
    bgm = BayesianGaussianMixture(
        n_components=n_comp, covariance_type='diag',
        weight_concentration_prior_type='dirichlet_distribution',
        weight_concentration_prior=1000,
        random_state=SEED, max_iter=500, n_init=1,
    )
    pred = bgm.fit_predict(X_pca)

    bin_counts = Counter(pred)
    small = {c for c, n in bin_counts.items() if n < MIN_BIN_SIZE}
    if small:
        pred = np.where(np.isin(pred, list(small)), -1, pred)

    cluster_results[(m, 'dpgmm')] = pred
    n_clusters = len(set(pred[pred != -1].tolist())) if (pred != -1).any() else 0
    assigned = int((pred != -1).sum())
    print(f'  {MODEL_LABELS[m]:>12s}: k={n_clusters:4d}  assigned={assigned:5d}/{N}')

# Copy DPGMM results for shared-mean models
for m in MODEL_NAMES:
    if m not in UNIQUE_MODELS:
        rep = GROUP_REP[MEAN_GROUPS[m]]
        cluster_results[(m, 'dpgmm')] = cluster_results[(rep, 'dpgmm')]
        print(f'  {MODEL_LABELS[m]:>12s}: copied from {MODEL_LABELS[rep]}')

print(f'\nTotal configurations: {len(cluster_results)}')
print(f'Unique runs: {len(UNIQUE_MODELS) * 3} (saved {(len(MODEL_NAMES) - len(UNIQUE_MODELS)) * 3} redundant runs)')

=== Greedy KMedoid ===
GreedyKMedoid: 1000 iterations, 582 clusters (after min_bin_size=5)
     NonLinear: k= 582  assigned=13434/17444
GreedyKMedoid: 1000 iterations, 560 clusters (after min_bin_size=5)
         Hinge: k= 560  assigned=12918/17444
GreedyKMedoid: 1000 iterations, 502 clusters (after min_bin_size=5)
       Bern-NT: k= 502  assigned=13146/17444


 Computing vMF similarity │██████████████████████████████│ 35/35 [00:02<00:00]


GreedyKMedoid: 1000 iterations, 465 clusters (after min_bin_size=5)
           PCL: k= 465  assigned=14732/17444
  UncertainGen: copied from NonLinear
    Bern-NT+UG: copied from Bern-NT
           LLA: copied from NonLinear

=== KMeans ===
     NonLinear: k= 494  assigned=17430/17444
         Hinge: k= 446  assigned=17440/17444
       Bern-NT: k= 487  assigned=17413/17444
           PCL: k= 492  assigned=17419/17444
  UncertainGen: copied from NonLinear
    Bern-NT+UG: copied from Bern-NT
           LLA: copied from NonLinear

=== DPGMM ===


e:\gnome\Lib\site-packages\sklearn\mixture\_base.py:293: ConvergenceWarning: Best performing initialization did not converge. Try different init parameters, or increase max_iter, tol, or check for degenerate data.
  warnings.warn(


     NonLinear: k= 338  assigned=17328/17444
         Hinge: k= 407  assigned=17345/17444
       Bern-NT: k= 425  assigned=17374/17444
           PCL: k= 397  assigned=17320/17444
  UncertainGen: copied from NonLinear
    Bern-NT+UG: copied from Bern-NT
           LLA: copied from NonLinear

Total configurations: 21
Unique runs: 12 (saved 9 redundant runs)


## 5. CheckM2 Evaluation

Run CheckM2 on each (model, clustering method) combination.
Each run exports bins as FASTA using **full-length contig sequences** → runs `checkm2 predict` → parses quality report.

In [ ]:
checkm2_results = {}
METHOD_LABELS = {'greedy_kmedoid': 'Greedy KMedoid', 'kmeans': 'KMeans', 'dpgmm': 'DPGMM'}
methods = ['greedy_kmedoid', 'kmeans', 'dpgmm']

# Run CheckM2 only for unique mean groups (4 models × 3 methods = 12 instead of 21)
for method in methods:
    for m in UNIQUE_MODELS:
        key = (m, method)
        pred = cluster_results[key]
        label = f'{MODEL_LABELS[m]} + {METHOD_LABELS[method]}'
        print(f'Running CheckM2: {label} ...', end=' ', flush=True)

        try:
            summary = evaluate_checkm2(
                labels=pred,
                sequences=full_sequences,
                threads=CHECKM2_THREADS,
                checkm2_cmd=CHECKM2_CMD,
                keep_files=False,
            )
            checkm2_results[key] = summary
            print(f'HQ={summary.n_high_quality}  MQ={summary.n_medium_quality}  '
                  f'LQ={summary.n_low_quality}  '
                  f'comp={summary.mean_completeness:.1f}%  '
                  f'cont={summary.mean_contamination:.1f}%')
        except Exception as ex:
            print(f'FAILED: {ex}')
            checkm2_results[key] = None

# Copy results for shared-mean models
for method in methods:
    for m in MODEL_NAMES:
        if m not in UNIQUE_MODELS:
            rep = GROUP_REP[MEAN_GROUPS[m]]
            checkm2_results[(m, method)] = checkm2_results[(rep, method)]
            s = checkm2_results[(m, method)]
            if s:
                print(f'{MODEL_LABELS[m]:>12s} + {METHOD_LABELS[method]}: '
                      f'copied from {MODEL_LABELS[rep]} '
                      f'(HQ={s.n_high_quality}, MQ={s.n_medium_quality})')

n_ok = sum(1 for v in checkm2_results.values() if v is not None)
print(f'\nDone. {n_ok}/{len(checkm2_results)} succeeded.')
print(f'Unique CheckM2 runs: {len(UNIQUE_MODELS) * len(methods)} '
      f'(saved {(len(MODEL_NAMES) - len(UNIQUE_MODELS)) * len(methods)} redundant runs)')

Running CheckM2: NonLinear + Greedy KMedoid ... HQ=3  MQ=54  LQ=525  comp=33.3%  cont=11.1%
Running CheckM2: Hinge + Greedy KMedoid ... HQ=5  MQ=54  LQ=501  comp=34.5%  cont=11.4%
Running CheckM2: Bern-NT + Greedy KMedoid ... HQ=3  MQ=56  LQ=443  comp=37.2%  cont=12.5%
Running CheckM2: PCL + Greedy KMedoid ... HQ=6  MQ=44  LQ=415  comp=37.4%  cont=14.3%
Running CheckM2: NonLinear + KMeans ... HQ=2  MQ=38  LQ=454  comp=54.9%  cont=23.5%
Running CheckM2: Hinge + KMeans ... HQ=4  MQ=42  LQ=400  comp=57.3%  cont=25.4%
Running CheckM2: Bern-NT + KMeans ... HQ=3  MQ=60  LQ=424  comp=55.6%  cont=22.5%
Running CheckM2: PCL + KMeans ... HQ=3  MQ=54  LQ=435  comp=54.7%  cont=24.1%
Running CheckM2: NonLinear + DPGMM ... HQ=2  MQ=35  LQ=301  comp=58.9%  cont=30.3%
Running CheckM2: Hinge + DPGMM ... HQ=4  MQ=42  LQ=361  comp=56.4%  cont=26.3%
Running CheckM2: Bern-NT + DPGMM ... HQ=2  MQ=69  LQ=354  comp=55.5%  cont=24.5%
Running CheckM2: PCL + DPGMM ... 

## 6. Results

In [ ]:
# ── Summary Table ──
methods = ['greedy_kmedoid', 'kmeans', 'dpgmm']

print(f'{"Model":>12s}  {"Method":>15s}  {"Bins":>5s}  {"Eval":>5s}  '
      f'{"HQ":>4s}  {"MQ":>4s}  {"LQ":>4s}  '
      f'{"Comp%":>6s}  {"Cont%":>6s}')
print('-' * 80)

for method in methods:
    for m in MODEL_NAMES:
        key = (m, method)
        s = checkm2_results.get(key)
        if s is None:
            print(f'{MODEL_LABELS[m]:>12s}  {METHOD_LABELS[method]:>15s}  {"FAIL":>5s}')
            continue
        print(f'{MODEL_LABELS[m]:>12s}  {METHOD_LABELS[method]:>15s}  '
              f'{s.n_bins_total:>5d}  {s.n_bins_evaluated:>5d}  '
              f'{s.n_high_quality:>4d}  {s.n_medium_quality:>4d}  {s.n_low_quality:>4d}  '
              f'{s.mean_completeness:>6.1f}  {s.mean_contamination:>6.1f}')
    print()

In [ ]:
# ── Bar Chart: High-Quality & Medium-Quality Bins ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

MODEL_COLORS = {
    'nl': 'gray', 'hinge': 'tab:purple', 'bern_nt': 'tab:olive',
    'ug': 'tab:red', 'ug_bnt': 'tab:cyan',
    'pcl': 'tab:blue', 'lla': 'tab:orange',
}

for ax, method in zip(axes, methods):
    hq_vals = []
    mq_vals = []
    labels_x = []
    colors = []
    for m in MODEL_NAMES:
        s = checkm2_results.get((m, method))
        hq_vals.append(s.n_high_quality if s else 0)
        mq_vals.append(s.n_medium_quality if s else 0)
        labels_x.append(MODEL_LABELS[m])
        colors.append(MODEL_COLORS[m])

    x = np.arange(len(MODEL_NAMES))
    ax.bar(x, hq_vals, color=colors, alpha=0.9, label='High Quality')
    ax.bar(x, mq_vals, bottom=hq_vals, color=colors, alpha=0.4, label='Medium Quality')
    ax.set_xticks(x)
    ax.set_xticklabels(labels_x, rotation=30, ha='right')
    ax.set_title(METHOD_LABELS[method])
    ax.set_ylabel('# Bins')
    ax.legend(['HQ (comp≥90, cont≤5)', 'MQ (comp≥50, cont≤10)'], fontsize=7)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Fecal Metagenome — CheckM2: HQ & MQ Bins by Model and Clustering', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Completeness vs Contamination scatter (best method by HQ count) ──
# Pick the method with the most HQ bins overall for the scatter plot
best_method = max(methods, key=lambda mt: sum(
    (checkm2_results.get((m, mt)) or CheckM2Summary()).n_high_quality
    for m in MODEL_NAMES))
print(f'Scatter plot method: {METHOD_LABELS[best_method]}')

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, m in enumerate(MODEL_NAMES):
    ax = axes[i]
    s = checkm2_results.get((m, best_method))
    if s is None or not s.per_bin:
        ax.set_title(f'{MODEL_LABELS[m]}: no data')
        continue

    comps = [b.completeness for b in s.per_bin]
    conts = [b.contamination for b in s.per_bin]
    tier_colors = {'high': 'green', 'medium': 'orange', 'low': 'red'}
    colors = [tier_colors[b.quality_tier] for b in s.per_bin]

    ax.scatter(conts, comps, c=colors, alpha=0.5, s=15, edgecolors='none')
    ax.axhline(90, color='green', ls='--', alpha=0.4, lw=0.8)
    ax.axhline(50, color='orange', ls='--', alpha=0.4, lw=0.8)
    ax.axvline(5, color='green', ls='--', alpha=0.4, lw=0.8)
    ax.axvline(10, color='orange', ls='--', alpha=0.4, lw=0.8)
    ax.set_xlabel('Contamination (%)')
    ax.set_ylabel('Completeness (%)')
    ax.set_title(f'{MODEL_LABELS[m]} (HQ={s.n_high_quality}, MQ={s.n_medium_quality})')
    ax.set_xlim(-2, 105)
    ax.set_ylim(-2, 105)
    ax.grid(alpha=0.2)

axes[-1].set_visible(False)

plt.suptitle(f'Fecal Metagenome — {METHOD_LABELS[best_method]}: Per-Bin Completeness vs Contamination', fontsize=13)
plt.tight_layout()
plt.show()

## 7. CheckM2 After Uncertainty Rejection (75% Coverage)

Cluster-then-reject: use the full-data clustering from Section 4, then reject the 25% most
uncertain contigs (by mean variance or 1/kappa). Re-apply `min_bin_size` to shrunk bins,
then run CheckM2 on the surviving bins.

Only uncertainty-aware models are meaningful here:
- **UncertainGen** (variance) — uses NL clusters
- **Bern-NT+UG** (variance) — uses Bern-NT clusters
- **LLA** (variance) — uses NL clusters
- **PCL** (1/kappa) — uses PCL clusters

Deterministic models (NL, Hinge, Bern-NT) have no uncertainty signal, so their baselines
from Section 5 serve as the no-rejection reference.

In [ ]:
COVERAGE = 0.75

# ── Uncertainty-aware models and their uncertainty scores ──
UNCERTAIN_MODELS = {
    'ug':     {'unc': contig_embs['ug'].variance.mean(axis=1),
               'clusters_from': 'nl'},
    'ug_bnt': {'unc': contig_embs['ug_bnt'].variance.mean(axis=1),
               'clusters_from': 'bern_nt'},
    'lla':    {'unc': contig_embs['lla'].variance.mean(axis=1),
               'clusters_from': 'nl'},
    'pcl':    {'unc': 1.0 / contig_embs['pcl'].kappa,
               'clusters_from': 'pcl'},
}

def apply_postcluster_rejection(full_pred, unc, coverage, min_bin_size):
    """Reject the most uncertain (1-coverage) fraction, re-apply min_bin_size."""
    N = len(unc)
    n_keep = max(min_bin_size + 1, int(round(N * coverage)))
    order_asc = np.argsort(unc)
    rejected_idx = order_asc[n_keep:]

    new_pred = full_pred.copy()
    new_pred[rejected_idx] = -1

    # Re-apply min_bin_size: clusters shrunk below threshold -> -1
    if (new_pred != -1).any():
        unique, counts = np.unique(new_pred[new_pred != -1], return_counts=True)
        for cl, cnt in zip(unique, counts):
            if cnt < min_bin_size:
                new_pred[new_pred == cl] = -1

    return new_pred

# ── Apply rejection and run CheckM2 ──
checkm2_rejected = {}  # (model, method) -> CheckM2Summary

for m, cfg in UNCERTAIN_MODELS.items():
    unc = cfg['unc']
    src = cfg['clusters_from']
    print(f'\n{"=" * 60}')
    print(f'{MODEL_LABELS[m]}  (clusters from {MODEL_LABELS[src]}, '
          f'coverage={COVERAGE:.0%})')
    print(f'{"=" * 60}')

    for method in methods:
        full_pred = cluster_results[(src, method)]
        pred_rej = apply_postcluster_rejection(full_pred, unc, COVERAGE, MIN_BIN_SIZE)

        actual_cov = (pred_rej != -1).sum() / len(pred_rej)
        n_clusters = len(set(pred_rej[pred_rej != -1].tolist())) if (pred_rej != -1).any() else 0

        label = f'{MODEL_LABELS[m]} + {METHOD_LABELS[method]}'
        print(f'  {label}: coverage={actual_cov:.1%}  bins={n_clusters}  ...', end=' ', flush=True)

        try:
            summary = evaluate_checkm2(
                labels=pred_rej,
                sequences=full_sequences,
                threads=CHECKM2_THREADS,
                checkm2_cmd=CHECKM2_CMD,
                keep_files=False,
            )
            checkm2_rejected[(m, method)] = summary
            print(f'HQ={summary.n_high_quality}  MQ={summary.n_medium_quality}  '
                  f'LQ={summary.n_low_quality}  '
                  f'comp={summary.mean_completeness:.1f}%  '
                  f'cont={summary.mean_contamination:.1f}%')
        except Exception as ex:
            print(f'FAILED: {ex}')
            checkm2_rejected[(m, method)] = None

n_ok = sum(1 for v in checkm2_rejected.values() if v is not None)
print(f'\nDone. {n_ok}/{len(checkm2_rejected)} succeeded.')

In [ ]:
# ── Comparison Table: Full vs 75% Coverage ──
# For each uncertainty model, show the baseline (deterministic, 100%) and rejected (75%) side by side

print(f'{"Model":>12s}  {"Method":>15s}  '
      f'{"HQ":>4s}  {"MQ":>4s}  {"Comp%":>6s}  {"Cont%":>6s}  │  '
      f'{"HQ":>4s}  {"MQ":>4s}  {"Comp%":>6s}  {"Cont%":>6s}  {"ΔHQ":>4s}  {"ΔMQ":>4s}')
print(f'{"":>12s}  {"":>15s}  '
      f'{"── 100% coverage (baseline) ──":>26s}          │  '
      f'{"── 75% coverage (rejected) ──":>26s}')
print('-' * 110)

for m in UNCERTAIN_MODELS:
    src = UNCERTAIN_MODELS[m]['clusters_from']
    for method in methods:
        s_full = checkm2_results.get((src, method))
        s_rej  = checkm2_rejected.get((m, method))

        if s_full is None or s_rej is None:
            continue

        dhq = s_rej.n_high_quality - s_full.n_high_quality
        dmq = s_rej.n_medium_quality - s_full.n_medium_quality
        dhq_s = f'{dhq:+d}'
        dmq_s = f'{dmq:+d}'

        print(f'{MODEL_LABELS[m]:>12s}  {METHOD_LABELS[method]:>15s}  '
              f'{s_full.n_high_quality:>4d}  {s_full.n_medium_quality:>4d}  '
              f'{s_full.mean_completeness:>6.1f}  {s_full.mean_contamination:>6.1f}  │  '
              f'{s_rej.n_high_quality:>4d}  {s_rej.n_medium_quality:>4d}  '
              f'{s_rej.mean_completeness:>6.1f}  {s_rej.mean_contamination:>6.1f}  '
              f'{dhq_s:>4s}  {dmq_s:>4s}')
    print()

In [ ]:
# ── Bar Chart: 100% vs 75% coverage (HQ+MQ) per clustering method ──
unc_models = list(UNCERTAIN_MODELS.keys())
unc_labels = [MODEL_LABELS[m] for m in unc_models]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, method in zip(axes, methods):
    x = np.arange(len(unc_models))
    w = 0.35

    hq_full, mq_full = [], []
    hq_rej, mq_rej = [], []
    for m in unc_models:
        src = UNCERTAIN_MODELS[m]['clusters_from']
        sf = checkm2_results.get((src, method))
        sr = checkm2_rejected.get((m, method))
        hq_full.append(sf.n_high_quality if sf else 0)
        mq_full.append(sf.n_medium_quality if sf else 0)
        hq_rej.append(sr.n_high_quality if sr else 0)
        mq_rej.append(sr.n_medium_quality if sr else 0)

    # 100% bars (left)
    ax.bar(x - w/2, hq_full, w, color='steelblue', alpha=0.9, label='HQ (100%)')
    ax.bar(x - w/2, mq_full, w, bottom=hq_full, color='steelblue', alpha=0.3, label='MQ (100%)')
    # 75% bars (right)
    ax.bar(x + w/2, hq_rej, w, color='coral', alpha=0.9, label='HQ (75%)')
    ax.bar(x + w/2, mq_rej, w, bottom=hq_rej, color='coral', alpha=0.3, label='MQ (75%)')

    ax.set_xticks(x)
    ax.set_xticklabels(unc_labels, rotation=30, ha='right')
    ax.set_title(METHOD_LABELS[method])
    ax.set_ylabel('# Bins')
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Fecal Metagenome — CheckM2: 100% vs {COVERAGE:.0%} Coverage (Uncertainty Rejection)',
             fontsize=13)
plt.tight_layout()
plt.show()